In [ ]:
import glob, os, functools
import numpy as np
import pandas as pd
import SimpleITK as sitk
from util.registration import nrrd_reg_rigid
from util.interpolate import interpolate
from util.bbox import get_bbox_3D
from util.data_util import get_arr_from_nrrd, get_bbox, generate_sitk_obj_from_npy_array
from scipy import ndimage
from SimpleITK.extra import GetArrayFromImage
from scipy import ndimage
import cv2


In [16]:
def normalize_data(patient_id, img, seg, crop_shape, return_type, output_img_dir, 
             output_seg_dir, image_format):

    # get image, arr, and spacing
    #image_object = sitk.ReadImage(img_dir)
    
    image_arr = sitk.GetArrayFromImage(img)
    #print(image_arr.shape)
    image_origin = img.GetOrigin()
    
    #label_object = sitk.ReadImage(seg_dir)
    label_arr = sitk.GetArrayFromImage(seg)
    
    label_origin = seg.GetOrigin()
    
    #assert image_arr.shape==label_arr.shape, "image & label shape do not match!"
    #print('max seg value:', np.max(label_arr))    
    # get center. considers all blobs
    bbox = get_bbox(label_arr)
    # returns center point of the label array bounding box
    Z, Y, X = int(bbox[9]), int(bbox[10]), int(bbox[11]) 
    #print('Original Centroid: ', X, Y, Z)
    
    #find origin translation from label to image
    #print('image origin: ', image_origin, 'label origin: ', label_origin)
    origin_dif = tuple(np.subtract(label_origin, image_origin).astype(int))
    #print('origin difference: ', origin_dif)
    
    X_shift, Y_shift, Z_shift = tuple(np.add((X, Y, Z), np.divide(origin_dif, (1, 1, 3)).astype(int)))
    #print('Centroid shifted:', X_shift, Y_shift, Z_shift) 
    #print(image_arr.shape)
    c, y, x = image_arr.shape
    
    ## Get center of mass to center the crop in Y plane
    mask_arr = np.copy(image_arr) 
    mask_arr[mask_arr > -500] = 1
    mask_arr[mask_arr <= -500] = 0
    mask_arr[mask_arr >= -500] = 1 
    #print('mask_arr min and max:', np.amin(mask_arr), np.amax(mask_arr))
    centermass = ndimage.measurements.center_of_mass(mask_arr) # z,x,y   
    cpoint = c - crop_shape[2]//2
    #print('cpoint, ', cpoint)
    centermass = ndimage.measurements.center_of_mass(mask_arr[cpoint, :, :])   
    #print('center of mass: ', centermass)
    startx = int(centermass[0] - crop_shape[0]//2)
    starty = int(centermass[1] - crop_shape[1]//2)      
    #startx = x//2 - crop_shape[0]//2       
    #starty = y//2 - crop_shape[1]//2
    startz = int(c - crop_shape[2])
    
    #---cut bottom slices---
    #image_arr = image_arr[30:, :, :]
    #label_arr = label_arr[30:, :, :]
    
    #-----normalize CT data signals-------
    norm_type = 'np_clip'
    #image_arr[image_arr <= -1024] = -1024
    ## strip skull, skull UHI = ~700
    #image_arr[image_arr > 700] = 0
    ## normalize UHI to 0 - 1, all signlas outside of [0, 1] will be 0;
    if norm_type == 'np_interp':
        image_arr = np.interp(image_arr, [-200, 200], [0, 1])
    elif norm_type == 'np_clip':
        image_arr = np.clip(image_arr, a_min=-175, a_max=275)
        MAX, MIN = image_arr.max(), image_arr.min()
        image_arr = (image_arr - MIN) / (MAX - MIN)
    
    # crop and pad array
    #z_seg_crop = int(crop_shape[2]*0.35)
    if startz < 0:
        image_arr = np.pad(
            image_arr,
            ((abs(startz)//2, abs(startz)//2), (0, 0), (0, 0)), 
            'constant', 
            constant_values=-1024)
        label_arr = np.pad(
            label_arr,
            ((abs(startz)//2, abs(startz)//2), (0, 0), (0, 0)), 
            'constant', 
            constant_values=0)
        image_arr_crop = image_arr[0:crop_shape[2], starty:starty+crop_shape[1], startx:startx+crop_shape[0]]
        label_arr_crop = label_arr[0:crop_shape[2], starty:starty+crop_shape[1], startx:startx+crop_shape[0]]
    elif (startx<0) :
        image_arr = np.pad(
            image_arr,
            ((0,0), (abs(startx), abs(startx)), (abs(startx), abs(startx))), 
            'constant', 
            constant_values=0)
        label_arr = np.pad(
            label_arr,
            ((0,0), (abs(startx), abs(startx)), (abs(startx), abs(startx))), 
            'constant', 
            constant_values=0)
        print(startx)
        image_arr_crop = image_arr[0:crop_shape[2], 0:crop_shape[1], 0:crop_shape[0]]
        label_arr_crop = label_arr[0:crop_shape[2], 0:crop_shape[1], 0:crop_shape[0]]
    
    else:
        image_arr_crop = image_arr[0:crop_shape[2], starty:starty+crop_shape[1], startx:startx+crop_shape[0]]
        label_arr_crop = label_arr[0:crop_shape[2], starty:starty+crop_shape[1], startx:startx+crop_shape[0]]
    
    # save nrrd
    output_img = output_img_dir + '/' + patient_id + '.' + image_format
    output_seg = output_seg_dir + '/' + patient_id + '.seg.' + image_format
    # save image
    print(image_arr_crop.shape)
    print(label_arr_crop.shape)
    img_sitk = sitk.GetImageFromArray(image_arr_crop)
    print(img_sitk.GetSpacing())
   
    
    #img_sitk.SetSpacing(img.GetSpacing())
    #img_sitk.SetOrigin(img.GetOrigin())
    print("final shape to be saved")
    print(img_sitk.GetSize())
    print(img_sitk.GetSpacing())
    
    writer = sitk.ImageFileWriter()
    writer.SetFileName(output_img)
    writer.SetUseCompression(True)
    writer.Execute(img_sitk)
    # save label
    seg_sitk = sitk.GetImageFromArray(label_arr_crop)
   
    #seg_sitk.SetSpacing(seg.GetSpacing())
    #seg_sitk.SetOrigin(seg.GetOrigin())
   
    print("final seg shape to be saved")
    print(seg_sitk.GetSize())
    print(seg_sitk.GetSpacing())
   
    writer = sitk.ImageFileWriter()
    writer.SetFileName(output_seg)
    writer.SetUseCompression(True)
    writer.Execute(seg_sitk)


In [18]:
proj_dir = '../data/raw-data'
image_format = 'nrrd'
img_raw_dir = proj_dir + '/scans'
seg_n_raw_dir = proj_dir + '/Expert-Segmenations'

img_crop_dir = proj_dir + '/scans-normalized'
seg_n_crop_dir = proj_dir + '/segs-normalized'
   
img_dirs = [i for i in sorted(glob.glob(img_raw_dir + '/*nrrd'))]
seg_n_dirs = [i for i in sorted(glob.glob(seg_n_raw_dir + '/*nrrd'))]
seg_dirs = seg_n_dirs
seg_crop_dir = seg_n_crop_dir
img_ids = []
bad_ids = []
bad_scans = []
count = 0
    
for img_dir in img_dirs:
    img_id = img_dir.split('/')[-1].split('.')[0]
        #print(img_id)
    for seg_dir in seg_dirs:
        seg_id = seg_dir.split('/')[-1].split('.')[0]
      
            #print(seg_id)
        if img_id == seg_id:
            img_ids.append(img_id)
            count += 1
            print(count, img_id)
            # load img and seg
            img = sitk.ReadImage(img_dir, sitk.sitkFloat32)
            seg = sitk.ReadImage(seg_dir, sitk.sitkFloat32)
            z_img = img.GetSize()[2]
            z_seg = seg.GetSize()[2]
            spacing = img.GetSpacing()[-1]
            if z_img < 10:
                print('This is an incomplete scan!')
                bad_scans.append(seg_id)
            else:
                try:
                    print('normalizing')
                    normalize_data(
                            patient_id=img_id,
                            img=img,
                            seg=seg,
                            crop_shape=(512,512,z_seg),
                            return_type='sitk_object',
                            output_img_dir=img_crop_dir,
                            output_seg_dir=seg_crop_dir,
                            image_format=image_format)
                    print('successfully crop!')
                        
                except Exception as e:
                    bad_ids.append(img_id)
                    print(img_id, e)
    print('bad ids:', bad_ids)
    print('incomplete scans:', bad_scans)






------start registration--------
1 mdacc_HNSCC-01-0001_CT-SIM-12-05-1998-_raw_raw_raw_xx
(512, 512, 117)
117
(0.9375, 0.9375, 2.5)
interpolate
cropping


/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_5510/3391285409.py:56: DeprecationWarning: Please use `center_of_mass` from the `scipy.ndimage` namespace, the `scipy.ndimage.measurements` namespace is deprecated.
  centermass = ndimage.measurements.center_of_mass(mask_arr) # z,x,y
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_5510/3391285409.py:59: DeprecationWarning: Please use `center_of_mass` from the `scipy.ndimage` namespace, the `scipy.ndimage.measurements` namespace is deprecated.
  centermass = ndimage.measurements.center_of_mass(mask_arr[cpoint, :, :])


(117, 256, 256)
(117, 256, 256)
(1.0, 1.0, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final shape to be saved
(512, 512, 117)
(0.5, 0.5, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final seg shape to be saved
(512, 512, 117)
(0.5, 0.5, 1.0)
successfully crop!
bad ids: []
incomplete scans: []
2 mdacc_HNSCC-01-0003_CT-SIM-11-20-2001-_raw_raw_raw_xx
(512, 512, 156)
156
(0.9765625, 0.9765625, 2.5)
interpolate
cropping
(156, 256, 256)
(156, 256, 256)
(1.0, 1.0, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final shape to be saved
(512, 512, 156)
(0.5, 0.5, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final seg shape to be saved
(512, 512, 156)
(0.5, 0.5, 1.0)
successfully crop!
bad ids: []
incomplete scans: []
3 mdacc_HNSCC-01-0004_CT-SIM-08-24-1996-_raw_raw_raw_xx
(512, 512, 130)
130
(0.9375, 0.9375, 3.0)
interpolate
cropping
(130, 256, 256)
(130, 256, 256)
(1.0, 1.0, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final shape

successfully crop!
bad ids: []
incomplete scans: []
20 mdacc_HNSCC-01-0020_CT-SIM-07-28-1998-_raw_raw_raw_xx
(512, 512, 132)
132
(0.9765625, 0.9765625, 3.0)
interpolate
cropping
(132, 256, 256)
(132, 256, 256)
(1.0, 1.0, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final shape to be saved
(512, 512, 132)
(0.5, 0.5, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final seg shape to be saved
(512, 512, 132)
(0.5, 0.5, 1.0)
successfully crop!
bad ids: []
incomplete scans: []
21 mdacc_HNSCC-01-0021_CT-SIM-09-01-1998-_raw_raw_raw_xx
(512, 512, 133)
133
(0.9765625, 0.9765625, 3.0)
interpolate
cropping
(133, 256, 256)
(133, 256, 256)
(1.0, 1.0, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final shape to be saved
(512, 512, 133)
(0.5, 0.5, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final seg shape to be saved
(512, 512, 133)
(0.5, 0.5, 1.0)
successfully crop!
bad ids: []
incomplete scans: []
22 mdacc_HNSCC-01-0022_CT-SIM-08-24-1998-_raw_raw_ra

cropping
(130, 256, 256)
(130, 256, 256)
(1.0, 1.0, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final shape to be saved
(512, 512, 130)
(0.5, 0.5, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final seg shape to be saved
(512, 512, 130)
(0.5, 0.5, 1.0)
successfully crop!
bad ids: []
incomplete scans: []
39 mdacc_HNSCC-01-0042_CT-SIM-01-19-1999-_raw_raw_raw_xx
(512, 512, 150)
150
(0.9375, 0.9375, 2.4300000000000637)
interpolate
cropping
(150, 256, 256)
(150, 256, 256)
(1.0, 1.0, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final shape to be saved
(512, 512, 150)
(0.5, 0.5, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final seg shape to be saved
(512, 512, 150)
(0.5, 0.5, 1.0)
successfully crop!
bad ids: []
incomplete scans: []
40 mdacc_HNSCC-01-0043_CT-SIM-12-15-1998-_raw_raw_raw_xx
(512, 512, 135)
135
(0.9765625, 0.9765625, 3.0)
interpolate
cropping
(135, 256, 256)
(135, 256, 256)
(1.0, 1.0, 1.0)
printing input_spacing in the resize
(

...

-3
(261, 256, 256)
(261, 256, 256)
(1.0, 1.0, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final shape to be saved
(512, 512, 261)
(0.5, 0.5, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final seg shape to be saved
(512, 512, 261)
(0.5, 0.5, 1.0)
successfully crop!
bad ids: []
incomplete scans: []
405 mdacc_HNSCC-01-0550_CT-SIM-10-10-2010-_raw_raw_raw_xx
(512, 512, 306)
306
(0.488281, 0.488281, 1.0)
interpolate
cropping
-3
(306, 256, 256)
(306, 256, 256)
(1.0, 1.0, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final shape to be saved
(512, 512, 306)
(0.5, 0.5, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final seg shape to be saved
(512, 512, 306)
(0.5, 0.5, 1.0)
successfully crop!
bad ids: []
incomplete scans: []
406 mdacc_HNSCC-01-0551_CT-SIM-10-10-2010-_raw_raw_raw_xx
(512, 512, 135)
135
(0.554, 0.554, 2.0)
interpolate
cropping
(135, 256, 256)
(135, 256, 256)
(1.0, 1.0, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final

printing input_spacing in the resize
(1.0, 1.0, 1.0)
final seg shape to be saved
(512, 512, 293)
(0.5, 0.5, 1.0)
successfully crop!
bad ids: []
incomplete scans: []
423 mdacc_HNSCC-01-0568_CT-SIM-11-13-2011-_raw_raw_raw_xx
(512, 512, 101)
101
(0.587891, 0.587891, 2.5)
interpolate
cropping
(101, 256, 256)
(101, 256, 256)
(1.0, 1.0, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final shape to be saved
(512, 512, 101)
(0.5, 0.5, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final seg shape to be saved
(512, 512, 101)
(0.5, 0.5, 1.0)
successfully crop!
bad ids: []
incomplete scans: []
bad ids: []
incomplete scans: []
424 mdacc_HNSCC-01-0570_CT-SIM-11-13-2011-_raw_raw_raw_xx
(512, 512, 328)
328
(0.488281, 0.488281, 1.0)
interpolate
cropping
-3
(328, 256, 256)
(328, 256, 256)
(1.0, 1.0, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final shape to be saved
(512, 512, 328)
(0.5, 0.5, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final seg shape t

successfully crop!
bad ids: []
incomplete scans: []
441 mdacc_HNSCC-01-0590_CT-SIM-11-13-2011-_raw_raw_raw_xx
(512, 512, 307)
307
(0.488281, 0.488281, 1.0)
interpolate
cropping
-3
(307, 256, 256)
(307, 256, 256)
(1.0, 1.0, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final shape to be saved
(512, 512, 307)
(0.5, 0.5, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final seg shape to be saved
(512, 512, 307)
(0.5, 0.5, 1.0)
successfully crop!
bad ids: []
incomplete scans: []
442 mdacc_HNSCC-01-0591_CT-SIM-11-13-2011-_raw_raw_raw_xx
(512, 512, 314)
314
(0.488281, 0.488281, 1.0)
interpolate
cropping
-3
(314, 256, 256)
(314, 256, 256)
(1.0, 1.0, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final shape to be saved
(512, 512, 314)
(0.5, 0.5, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final seg shape to be saved
(512, 512, 314)
(0.5, 0.5, 1.0)
successfully crop!
bad ids: []
incomplete scans: []
443 mdacc_HNSCC-01-0592_CT-SIM-12-16-2012-_raw_r

cropping
-3
(281, 256, 256)
(281, 256, 256)
(1.0, 1.0, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final shape to be saved
(512, 512, 281)
(0.5, 0.5, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final seg shape to be saved
(512, 512, 281)
(0.5, 0.5, 1.0)
successfully crop!
bad ids: []
incomplete scans: []
460 mdacc_HNSCC-01-0610_CT-SIM-12-16-2012-_raw_raw_raw_xx
(512, 512, 288)
288
(0.488281, 0.488281, 1.0)
interpolate
cropping
-3
(288, 256, 256)
(288, 256, 256)
(1.0, 1.0, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final shape to be saved
(512, 512, 288)
(0.5, 0.5, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final seg shape to be saved
(512, 512, 288)
(0.5, 0.5, 1.0)
successfully crop!
bad ids: []
incomplete scans: []
461 mdacc_HNSCC-01-0611_CT-SIM-12-16-2012-_raw_raw_raw_xx
(512, 512, 230)
230
(0.488281, 0.488281, 1.0)
interpolate
cropping
-3
(230, 256, 256)
(230, 256, 256)
(1.0, 1.0, 1.0)
printing input_spacing in the resize
(1.

printing input_spacing in the resize
(1.0, 1.0, 1.0)
final seg shape to be saved
(512, 512, 108)
(0.5, 0.5, 1.0)
successfully crop!
bad ids: []
incomplete scans: []
478 mdacc_HNSCC-01-0629_CT-SIM-12-16-2012-_raw_raw_raw_xx
(512, 512, 280)
280
(0.488281, 0.488281, 1.0)
interpolate
cropping
-3
(280, 256, 256)
(280, 256, 256)
(1.0, 1.0, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final shape to be saved
(512, 512, 280)
(0.5, 0.5, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final seg shape to be saved
(512, 512, 280)
(0.5, 0.5, 1.0)
successfully crop!
bad ids: []
incomplete scans: []
479 mdacc_HNSCC-01-0630_CT-SIM-12-16-2012-_raw_raw_raw_xx
(512, 512, 255)
255
(0.488281, 0.488281, 1.0)
interpolate
cropping
-3
(255, 256, 256)
(255, 256, 256)
(1.0, 1.0, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final shape to be saved
(512, 512, 255)
(0.5, 0.5, 1.0)
printing input_spacing in the resize
(1.0, 1.0, 1.0)
final seg shape to be saved
(512, 512, 255)
(0.